In [1]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import torch
from IPython.display import *


from pacer import (
    # read_dat_file,
    DatVersion,
    RawGPSSource,
    GPMFSource,
    SequentialGPSSource,
    CoordinateSystem,
    GPSSample,
    Vec3f,
    PointInTime_GPSSample,
    Point,
    Laps,
    Segment,
    Lap,
)

In [2]:
# vector helpers (pacer.Point exposes only x, y, rot -- no operators / scalar)
def vadd(p, q):   return Point(p.x + q.x, p.y + q.y)
def vsub(p, q):   return Point(p.x - q.x, p.y - q.y)
def vscale(s, p): return Point(s * p.x, s * p.y)
def vdot(p, q):   return p.x * q.x + p.y * q.y

In [3]:
files = [
    "/Users/daniil/Desktop/D24/GX010060.MP4",
]

gpmf = GPMFSource(files[0])
# for f in files[1:]:
#     gpmf = SequentialGPSSource(gpmf, GPMFSource(f))

gpmf.get_total_duration()
samples = []


def on_sample(s: GPSSample, _, _2):
    if s.full_speed > 3:
        samples.append((s, gpmf.current_time_span()))


while not gpmf.is_end():
    gpmf.read_samples(on_sample)
    gpmf.next()

In [4]:
s1, s2 = samples[0][0], samples[531][0]
print(f"Sample 1: {s1}")
print(f"Sample 2: {s2}")

Sample 1: GPSSample(lat=52.040054, lon=-0.785116, altitude=70.778000, full_speed=3.290000, ground_speed=4.361000)
Sample 2: GPSSample(lat=52.039415, lon=-0.781324, altitude=73.886000, full_speed=10.550000, ground_speed=10.533000)


In [5]:
cs = CoordinateSystem(s1)

In [6]:
rough_frequency = len(samples) / len(set(span for _, span in samples))

data = np.array(
    [
        cs.distance(s1, s2)
        / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
        * rough_frequency
        for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
    ]
)

px.scatter(data)

In [7]:
di = np.round(
    np.array(
        [
            cs.distance(s1, s2)
            / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
            * rough_frequency
            for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
        ]
    )
)

px.scatter(di)

In [8]:
floor = torch.Tensor([b for (_, (b, _)) in samples])
ceil = torch.Tensor([e for (_, (_, e)) in samples])
di = np.round(
    np.array(
        [
            cs.distance(s1, s2)
            / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
            * rough_frequency
            for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
        ]
    )
)
# clamp zero-distance steps so the loss never divides by zero (was -> NaN)
di = torch.Tensor(np.concatenate([[1], np.maximum(di, 1)]))

assert ceil.shape == floor.shape == di.shape


def loss(x):
    assert x.shape == di.shape, f"Expected {di.shape}, got {x.shape}"

    my_diffs = x[1:] - x[:-1]
    my_diffs = my_diffs / di[1:]
    spacing = ((my_diffs - my_diffs.mean()) ** 2).mean()
    constraints = (((floor - x).clip(min=0) + (x - ceil).clip(min=0)) ** 2).mean()
    return spacing + constraints

In [9]:
t1 = (floor + ceil) / 2
t1.requires_grad_()

learning_curve = []

for lr in [1e-1, 1e-2, 1e-3]:
    optimizer = torch.optim.Adam([t1], lr=lr)
    for i in range(100):
        optimizer.zero_grad()
        l = loss(t1)
        l.backward()
        optimizer.step()
        learning_curve.append((lr, i, l.item()))
px.line(
    pd.DataFrame(learning_curve, columns=["lr", "iteration", "loss"]),
    y="loss",
    log_y=True,
    color="lr",
    title="Learning curve",
)

In [10]:
phase = torch.tensor(floor[0], requires_grad=True)
frequency = torch.tensor(rough_frequency, requires_grad=True)

t2 = phase + 1 / frequency * (di.long().cumsum(0).float() - 1)
learning_curve = []

for lr in [1e-1, 1e-2, 1e-3]:
    optimizer = torch.optim.Adam([phase, frequency], lr=lr)
    for i in range(100):
        optimizer.zero_grad()
        l = loss(t2)
        l.backward(retain_graph=True)
        optimizer.step()
        t2 = phase + 1 / frequency * (di.long().cumsum(0).float() - 1)
        learning_curve.append((lr, i, l.item()))

px.line(
    pd.DataFrame(learning_curve, columns=["lr", "iteration", "loss"]),
    y="loss",
    log_y=True,
    color="lr",
    title="Learning curve",
)

/var/folders/tj/1bthqtnj70nckrpyx8_z6qlr0000gn/T/ipykernel_46955/2896287337.py:1: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [11]:
t1n = t1.detach().numpy()
din = di.detach().numpy()
dt = t1n[1:] - t1n[:-1]
implied_frequency = din[1:] / np.where(dt > 1e-6, dt, np.nan)  # mask the few time reversals
px.scatter(
    implied_frequency,
    title="Implied GPS frequency from t1 (free interpolation)",
    labels={"value": "Hz", "index": "sample"},
).update_yaxes(range=[0, 40])

In [12]:
implied_frequency = di[1:] / (t2[1:] - t2[:-1])
px.scatter(
    implied_frequency.detach().numpy(),
    title="Implied GPS frequency from t2 (parametric — constant by design)",
    labels={"value": "Hz", "index": "sample"},
).update_yaxes(range=[0, 40])

In [13]:
laps = Laps()
laps.set_coordinate_system(cs)

for (s, span), t in zip(samples, t1):
    laps.add_point(s, t)

laps.sectors.start_line = laps.pick_random_start()
laps.update()

_lt = np.array([laps.lap_time(i) for i in range(laps.laps_count())])
_med = np.median(_lt[_lt > 1])
pd.DataFrame(
    [
        dict(
            lap=i,
            lap_time=round(float(laps.lap_time(i)), 3),
            clean=bool(0.7 * _med < laps.lap_time(i) < 1.3 * _med),
        )
        for i in range(laps.laps_count())
    ]
)

,lap,lap_time,clean
0,0,136.298,False
1,1,5.705,False
2,2,139.286,False
3,3,69.774,True
4,4,70.399,True
5,5,68.684,True
6,6,70.126,True
7,7,69.547,True
8,8,68.483,True
9,9,69.442,True


In [14]:
SMOOTH = 9  # GPS-track smoothing window (samples): suppresses position jitter.
            # See notebooks/noise-investigation.ipynb for the noise budget / before-after.


def _smooth(a, w=SMOOTH):
    a = np.asarray(a, float)
    return a if (w < 2 or len(a) < w) else np.convolve(a, np.ones(w) / w, "same")


def delta_table(lp, grid_n=250):
    """Delta vs the fastest clean lap on a common distance grid, with the GPS
    track smoothed first (np.interp alignment; no resample)."""
    lt = np.array([lp.lap_time(i) for i in range(lp.laps_count())])
    med = np.median(lt[lt > 1])
    clean = [i for i in range(lp.laps_count()) if 0.7 * med < lt[i] < 1.3 * med]
    if not clean:
        clean = [i for i in range(lp.laps_count()) if lt[i] > 1]

    def curve(i):
        L = lp.get_lap(i)
        t = np.array([L.points[j].time for j in range(len(L.points))])
        loc = np.array([[(q := cs.local(L.points[j].point)).x, q.y] for j in range(len(L.points))])
        x, y = _smooth(loc[:, 0]), _smooth(loc[:, 1])
        d = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x), np.diff(y)))])
        lat = _smooth([L.points[j].point.lat for j in range(len(L.points))])
        lon = _smooth([L.points[j].point.lon for j in range(len(L.points))])
        return d, t - t[0], lat, lon

    best = min(clean, key=lambda i: lt[i])
    db, tb, _, _ = curve(best)
    grid = np.linspace(0, db[-1], grid_n)
    best_t = np.interp(grid, db, tb)

    rows = []
    for i in clean:
        d, t, la, lo = curve(i)
        rows.append(pd.DataFrame(dict(
            distance=grid,
            delta=np.interp(grid, d, t) - best_t,
            lat=np.interp(grid, d, la),
            lon=np.interp(grid, d, lo),
            lap=i,
        )))
    return pd.concat(rows, ignore_index=True), best, float(lt[best]), clean


delta_by_lap, best, best_time, clean = delta_table(laps)
delta = delta_by_lap[delta_by_lap["lap"] == clean[-1]]["delta"].to_numpy()

px.line(
    delta_by_lap,
    x="distance",
    y="delta",
    color=delta_by_lap["lap"].astype(str),
    title=f"Time delta vs fastest clean lap (Lap {best} = {best_time:.2f}s, GPS-smoothed)",
    labels={"color": "lap", "delta": "delta vs best (s)", "distance": "distance (m)"},
)

In [15]:
lap_for_map = max(
    clean, key=lambda i: float(delta_by_lap[delta_by_lap["lap"] == i]["delta"].abs().max())
)
sub = delta_by_lap[delta_by_lap["lap"] == lap_for_map].copy()
sub["d_delta"] = sub["delta"].diff().rolling(10, min_periods=1).mean()

px.scatter_map(
    sub,
    lat="lat",
    lon="lon",
    color="d_delta",
    zoom=15,
    color_continuous_scale="RdBu_r",
    title=f"Lap {lap_for_map}: where time is lost (red) / gained (blue) vs best",
).update_layout(height=700)

In [16]:
px.histogram(
    delta_by_lap, x="delta", nbins=120,
    title="Distribution of per-point time delta vs best lap",
    labels={"delta": "delta vs best (s)"},
)

In [17]:
lap = laps.get_lap(1)


def build_lap_df(lap):
    n = min(len(lap.points), len(lap.cum_distances))
    return pd.DataFrame(
        [
            dict(
                lat=lap.points[i].point.lat,
                lon=lap.points[i].point.lon,
                time=lap.points[i].time - lap.points[0].time,
                distance=lap.cum_distances[i],
                i_point=i,
            )
            for i in range(n)
        ]
    )


all_laps = pd.concat(
    [build_lap_df(laps.get_lap(i)).assign(i_lap=i) for i in range(laps.laps_count())]
)
all_laps

,lat,lon,time,distance,i_point,i_lap
0,52.039752,-0.783212,0.000000,0.000000,0,0
1,52.039747,-0.783207,0.043486,0.696424,1,0
2,52.039734,-0.783191,0.142287,2.526712,2,0
3,52.039721,-0.783174,0.241118,4.349569,3,0
4,52.039709,-0.783157,0.339934,6.175767,4,0
...,...,...,...,...,...,...
707,52.039786,-0.783219,70.636352,1103.384535,707,20
708,52.039774,-0.783200,70.735229,1105.207797,708,20
709,52.039766,-0.783187,70.808392,1106.516253,709,20
0,52.039766,-0.783187,0.000000,0.000000,0,21


In [18]:
fig = px.scatter_map(
    all_laps,
    lat="lat",
    lon="lon",
    color="distance",
    hover_data=["i_lap", "i_point"],
    # map_style="basic",
    zoom=17,
)
fig.update_layout(height=800)


In [19]:
start_line = laps.sectors.start_line
p1, p2 = start_line.first, start_line.second

In [20]:
s1, s2 = map(lambda p: cs.global_(Vec3f(p.x, p.y, 0)), (p1, p2))
s1, s2

(GPSSample(lat=52.039730, lon=-0.783255, altitude=70.779374, full_speed=0.000000, ground_speed=0.000000),
 GPSSample(lat=52.039788, lon=-0.783144, altitude=70.779498, full_speed=0.000000, ground_speed=0.000000))

In [21]:
def add_segment(fig: go.Figure, s1: GPSSample, s2: GPSSample, name: str):
    fig.add_trace(
        go.Scattermap(
            mode="lines",
            lon=[s1.lon, s2.lon],
            lat=[s1.lat, s2.lat],
            name=name,
        )
    )
    return fig


add_segment(fig, s1, s2, "start_line")

In [22]:
def to_point(s: GPSSample):
    v = cs.local(s)
    return Point(v.x, v.y)


first_seg = Segment(
    to_point(laps.get_lap(0).points[0].point), to_point(laps.get_lap(0).points[1].point)
)

add_segment(
    fig, laps.get_lap(0).points[0].point, laps.get_lap(0).points[1].point, "first_one"
)

In [23]:
a, b = first_seg.first, first_seg.second
x, y = start_line.first, start_line.second
nrm = Point(-(x.y - y.y), x.x - y.x)  # perpendicular to the start line
da = vdot(nrm, vsub(a, x))
db = vdot(nrm, vsub(b, x))
ratio = da / (da - db) if (da - db) != 0 else 0.5
ratio

-0.030370282339245953

In [24]:
df = pd.DataFrame(
    dict(
        x=[
            start_line.first.x,
            start_line.second.x,
            None,
            first_seg.first.x,
            first_seg.second.x,
        ],
        y=[
            start_line.first.y,
            start_line.second.y,
            None,
            first_seg.first.y,
            first_seg.second.y,
        ],
        name=[1, 1, None, 2, 3],
    )
)

fig = px.line(df, x="x", y="y", hover_data="name")
fig

In [25]:
r = ratio if ratio is not None else 0.5
res = vadd(vscale(1 - r, a), vscale(r, b))
fig.add_trace(go.Scatter(x=[res.x], y=[res.y]))

In [26]:
points_df = pd.DataFrame(
    {
        "latitude": [s.lat for s, _ in samples],
        "longitude": [s.lon for s, _ in samples],
        # cap physically-impossible GPS speed glitches so the color scale stays useful
        "speed": np.clip([s.full_speed * 3.6 for s, _ in samples], 0, 100),
    }
)

px.scatter_map(
    points_df,
    lat="latitude",
    lon="longitude",
    color="speed",
    zoom=14,
    title="All GPS samples (colored by speed, km/h; glitches capped at 100)",
).update_layout(height=800)

In [27]:
delta_by_lap

,distance,delta,lat,lon,lap
0,0.000000,0.000000,28.910967,-0.435087,3
1,4.721824,0.023088,30.627487,-0.460917,3
2,9.443648,0.046175,32.344006,-0.486748,3
3,14.165473,0.069263,34.060525,-0.512578,3
4,18.887297,0.076703,35.764869,-0.538225,3
...,...,...,...,...,...
4495,1156.846929,2.192654,45.689586,-0.687668,20
4496,1161.568754,2.192478,43.745081,-0.658399,20
4497,1166.290578,2.193988,41.800575,-0.629130,20
4498,1171.012402,2.195403,39.862369,-0.599957,20


In [28]:
px.scatter(delta[1:] - delta[:-1])

In [29]:
c = 0.1
noise = np.round((delta[1:] - delta[:-1]) / c) * c
px.scatter(noise)

In [30]:
px.scatter(np.cumsum(noise), title="Cumulative noise")

In [31]:
px.line(delta - np.concatenate([[0], np.cumsum(noise)]))

In [32]:
laps_t2 = Laps()
laps_t2.set_coordinate_system(cs)
for (s, span), t in zip(samples, t2):
    laps_t2.add_point(s, float(t))
laps_t2.sectors.start_line = laps_t2.pick_random_start()
laps_t2.update()

delta_t2, best_t2, best_time_t2, _ = delta_table(laps_t2)
px.line(
    delta_t2,
    x="distance",
    y="delta",
    color=delta_t2["lap"].astype(str),
    title=f"Time delta with parametric (t2) interpolation vs Lap {best_t2} = {best_time_t2:.2f}s",
    labels={"color": "lap", "delta": "delta vs best (s)", "distance": "distance (m)"},
)